# AIP/DIP LSTM-attention model


## 1. Model definitions


In [ ]:
"""AIP/DIP LSTM-attention model."""

from __future__ import annotations

import argparse
import json
import math
import pickle
import random
import warnings
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, Dataset


# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

SEED = 42
MAX_WINDOWS = 12
SAS_REFERENCE_DATE = pd.Timestamp("1960-01-01")

ID_COL = "JID"
AGE_COL = "AGE"
SEX_COL = "SEX"
INSURANCE_COL = "INSUP"
OUTCOME_COL = "OUTCOME_DIP"
SEQUENTIAL_COL = "WINDOW_PKR_BB_DDD"
WINDOW_START_COL = "WINDOW_START"
WINDOW_END_COL = "WINDOW_END"
INDEX_DATE_COL = "INDEX_DN"
AIP_DATE_COL = "DIP_DN"

# Comorbidities
COMORBIDITIES = [
    "COPD",
    "DEMENTIA",
    "DEPRESS",
    "DM",
    "ESRD",
    "GOUT",
    "HTN",
    "LIPID",
    "LIVER",
    "OA",
    "OP",
    "STROKE",
    "TBI",
]

CONCOMITANT_MEDICATIONS = [
    "AGI",
    "ANTICONV",
    "ANTIDEP",
    "ACEI",
    "ARB",
    "BB",
    "BZD",
    "CCB_DHP",
    "CCB_NDHP",
    "DPP4",
    "GLP1",
    "INSULIN",
    "LOOP",
    "MEG",
    "MET",
    "SGLT2",
    "STATIN",
    "SU",
]


@dataclass(frozen=True)
class ModelConfig:
    hidden_dim: int = 50
    num_layers: int = 2
    learning_rate: float = 0.001
    epochs: int = 30
    batch_size: int = 32


@dataclass
class PatientArrays:
    patient_ids: np.ndarray                 # (N,)
    x_static_raw: np.ndarray                # (N, P), AGE remains unscaled here
    x_sequential_raw: np.ndarray            # (N, T, 1), padded values are 0
    y_sequence: np.ndarray                  # (N, T), padded values are 0
    mask: np.ndarray                        # (N, T), True only for valid at-risk windows
    y_overall: np.ndarray                   # (N,), any AIP within valid horizon
    valid_lengths: np.ndarray               # (N,)
    static_features: List[str]

    def __len__(self) -> int:
        return len(self.patient_ids)


@dataclass
class FittedScalers:
    age_scaler: MinMaxScaler
    sequential_scaler: MinMaxScaler
    age_index: int


@dataclass
class EvaluationResult:
    metrics: Dict[str, float]
    y_true: np.ndarray
    y_prob: np.ndarray
    y_pred: np.ndarray
    timestep_prob: np.ndarray
    y_sequence: np.ndarray
    mask: np.ndarray
    window_metrics: pd.DataFrame
    roc_table: pd.DataFrame
    pr_table: pd.DataFrame
    calibration_table: pd.DataFrame


# -----------------------------------------------------------------------------
# General utilities
# -----------------------------------------------------------------------------

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_table(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    print(f"Loading: {path}")

    if suffix == ".sas7bdat":
        try:
            df = pd.read_sas(path, format="sas7bdat", encoding="utf-8")
        except Exception as exc:
            warnings.warn(
                f"pandas.read_sas failed ({exc!r}); trying sas7bdat package fallback."
            )
            try:
                from sas7bdat import SAS7BDAT  # type: ignore
            except ImportError as import_exc:
                raise RuntimeError(
                    "Could not read SAS7BDAT with pandas, and package 'sas7bdat' is not installed."
                ) from import_exc
            with SAS7BDAT(str(path)) as reader:
                df = reader.to_data_frame()
    elif suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported input format: {suffix}")

    # Defensive decoding for SAS character columns returned as bytes.
    for col in df.select_dtypes(include=["object"]).columns:
        if df[col].map(lambda x: isinstance(x, (bytes, bytearray))).any():
            df[col] = df[col].map(
                lambda x: x.decode("utf-8", errors="replace")
                if isinstance(x, (bytes, bytearray))
                else x
            )

    print(f"Loaded {len(df):,} rows and {df.shape[1]:,} columns")
    return df


def convert_sas_date_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Convert SAS numeric dates (days since 1960-01-01) when present."""
    df = df.copy()
    for col in [INDEX_DATE_COL, WINDOW_START_COL, WINDOW_END_COL, AIP_DATE_COL]:
        if col not in df.columns:
            continue
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = SAS_REFERENCE_DATE + pd.to_timedelta(df[col], unit="D")
    return df


def _normalize_code_value(x: object) -> object:
    if pd.isna(x):
        return np.nan
    if isinstance(x, (np.integer, int)):
        return int(x)
    if isinstance(x, (np.floating, float)) and float(x).is_integer():
        return int(x)
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s


def map_binary_code(
    series: pd.Series,
    mapping: Dict[object, int],
    name: str,
    allow_existing_binary: bool = True,
) -> pd.Series:
    normalized = series.map(_normalize_code_value)
    nonmissing = set(normalized.dropna().unique().tolist())

    if allow_existing_binary and nonmissing.issubset({0, 1, "0", "1"}):
        result = normalized.map({0: 0, 1: 1, "0": 0, "1": 1})
    else:
        expanded: Dict[object, int] = {}
        for key, value in mapping.items():
            expanded[key] = value
            expanded[str(key)] = value
        result = normalized.map(expanded)

    if result.isna().any():
        bad = normalized[result.isna()].dropna().unique()[:10]
        raise ValueError(f"Unexpected/unmapped values in {name}: {bad}")
    return result.astype(np.int64)


def get_binary_features() -> List[str]:
    return COMORBIDITIES.copy() + CONCOMITANT_MEDICATIONS


def preprocess_dataframe(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    df = convert_sas_date_columns(df)
    binary_features = get_binary_features()
    static_features = [INSURANCE_COL, SEX_COL, AGE_COL] + binary_features

    required = [ID_COL, AGE_COL, SEX_COL, INSURANCE_COL, OUTCOME_COL, SEQUENTIAL_COL] + binary_features
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError("Missing required columns: " + ", ".join(missing))

    df = df.copy()
    df[INSURANCE_COL] = map_binary_code(
        df[INSURANCE_COL], mapping={4: 0, 5: 1}, name=INSURANCE_COL
    )
    df[SEX_COL] = map_binary_code(
        df[SEX_COL], mapping={1: 0, 2: 1}, name=SEX_COL
    )

    for col in binary_features + [OUTCOME_COL]:
        if df[col].isna().any():
            raise ValueError(f"Missing values found in required binary column: {col}")
        values = pd.to_numeric(df[col], errors="raise").astype(int)
        bad = set(values.unique()) - {0, 1}
        if bad:
            raise ValueError(f"Column {col} is not binary 0/1. Unexpected values: {sorted(bad)}")
        df[col] = values

    df[AGE_COL] = pd.to_numeric(df[AGE_COL], errors="raise").astype(float)
    df[SEQUENTIAL_COL] = pd.to_numeric(df[SEQUENTIAL_COL], errors="raise").astype(float)

    if df[[AGE_COL, SEQUENTIAL_COL]].isna().any().any():
        raise ValueError("Missing values found in AGE or sequential exposure variable.")

    return df, static_features


# -----------------------------------------------------------------------------
# Sequence construction and censoring
# -----------------------------------------------------------------------------

def _patient_sort_columns(group: pd.DataFrame) -> List[str]:
    for candidate in [WINDOW_START_COL, WINDOW_END_COL]:
        if candidate in group.columns:
            return [candidate]
    return []


def build_patient_arrays(
    df: pd.DataFrame,
    static_features: List[str],
    max_windows: int = MAX_WINDOWS,
    harmonize_non_events: bool = False,
    seed: int = SEED,
) -> PatientArrays:
    """
    Build one patient-level sequence per JID.

    - Includes the incident AIP window.
    - Excludes all post-event windows.
    - Pads remaining positions to max_windows with zeros.
    - mask=True only for genuine at-risk windows.
    - Optionally truncates non-event sequences to sampled AIP sequence lengths.

    If sequence-length harmonization was already performed upstream in SAS, leave
    harmonize_non_events=False to avoid truncating the data twice.
    """
    patient_records: List[dict] = []
    multiple_event_rows = 0
    overlong_sequences = 0

    # sort=False preserves incoming patient order while avoiding repeated df[df[id]==...] scans.
    for patient_id, group in df.groupby(ID_COL, sort=False):
        sort_cols = _patient_sort_columns(group)
        if sort_cols:
            group = group.sort_values(sort_cols, kind="stable")

        if len(group) > max_windows:
            overlong_sequences += 1
            group = group.iloc[:max_windows].copy()
        else:
            group = group.copy()

        outcomes = group[OUTCOME_COL].to_numpy(dtype=np.int64)
        event_positions = np.flatnonzero(outcomes == 1)
        has_event = len(event_positions) > 0
        if len(event_positions) > 1:
            multiple_event_rows += 1

        if has_event:
            # Event-triggered censoring: keep first event window, remove everything after it.
            valid_len = int(event_positions[0]) + 1
            group = group.iloc[:valid_len].copy()
            outcomes = group[OUTCOME_COL].to_numpy(dtype=np.int64)
        else:
            valid_len = len(group)

        if valid_len < 1:
            continue

        patient_records.append(
            {
                "patient_id": patient_id,
                "group": group,
                "has_event": int(has_event),
                "valid_len": valid_len,
            }
        )

    if not patient_records:
        raise ValueError("No usable patient sequences were constructed.")

    if overlong_sequences:
        warnings.warn(
            f"{overlong_sequences:,} patients had >{max_windows} rows; only the first "
            f"{max_windows} windows were retained. Verify upstream window construction."
        )
    if multiple_event_rows:
        warnings.warn(
            f"{multiple_event_rows:,} patients had multiple positive OUTCOME_DIP rows; "
            "the first event was used and later rows were censored."
        )

    if harmonize_non_events:
        rng = np.random.default_rng(seed)
        event_lengths = np.array(
            [r["valid_len"] for r in patient_records if r["has_event"] == 1], dtype=int
        )
        if len(event_lengths) == 0:
            raise ValueError("Cannot harmonize non-event lengths because there are no AIP events.")

        for record in patient_records:
            if record["has_event"] == 0:
                target_len = int(rng.choice(event_lengths))
                new_len = min(record["valid_len"], target_len, max_windows)
                record["group"] = record["group"].iloc[:new_len].copy()
                record["valid_len"] = new_len

    n = len(patient_records)
    p = len(static_features)
    x_static = np.zeros((n, p), dtype=np.float32)
    x_seq = np.zeros((n, max_windows, 1), dtype=np.float32)
    y_seq = np.zeros((n, max_windows), dtype=np.float32)
    mask = np.zeros((n, max_windows), dtype=bool)
    patient_ids = np.empty(n, dtype=object)
    valid_lengths = np.zeros(n, dtype=np.int64)

    for i, record in enumerate(patient_records):
        group = record["group"]
        valid_len = min(int(record["valid_len"]), max_windows)
        patient_ids[i] = record["patient_id"]
        valid_lengths[i] = valid_len

        x_static[i, :] = group.iloc[0][static_features].to_numpy(dtype=np.float32)
        x_seq[i, :valid_len, 0] = group.iloc[:valid_len][SEQUENTIAL_COL].to_numpy(dtype=np.float32)
        y_seq[i, :valid_len] = group.iloc[:valid_len][OUTCOME_COL].to_numpy(dtype=np.float32)
        mask[i, :valid_len] = True

    y_overall = (y_seq.max(axis=1) > 0).astype(np.float32)

    # Data checks
    event_counts_per_patient = y_seq.sum(axis=1)
    if np.any(event_counts_per_patient > 1):
        raise AssertionError("Post-censoring sequences contain more than one incident AIP event.")
    if np.any((y_seq > 0) & (~mask)):
        raise AssertionError("Positive outcomes found in padded/censored positions.")
    if not np.all(mask.sum(axis=1) == valid_lengths):
        raise AssertionError("Mask and valid_lengths are inconsistent.")

    return PatientArrays(
        patient_ids=patient_ids,
        x_static_raw=x_static,
        x_sequential_raw=x_seq,
        y_sequence=y_seq,
        mask=mask,
        y_overall=y_overall,
        valid_lengths=valid_lengths,
        static_features=static_features,
    )


def sequence_length_summary(data: PatientArrays) -> pd.DataFrame:
    rows = []
    for outcome_value, outcome_name in [(0, "non-AIP"), (1, "AIP")]:
        subset = data.valid_lengths[data.y_overall.astype(int) == outcome_value]
        counts = pd.Series(subset).value_counts().sort_index()
        denom = max(len(subset), 1)
        for length in range(1, MAX_WINDOWS + 1):
            count = int(counts.get(length, 0))
            rows.append(
                {
                    "group": outcome_name,
                    "window_length": length,
                    "n_patients": count,
                    "proportion": count / denom,
                }
            )
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# Train-only normalization
# -----------------------------------------------------------------------------

def fit_scalers(data: PatientArrays, indices: np.ndarray) -> FittedScalers:
    age_index = data.static_features.index(AGE_COL)
    age_scaler = MinMaxScaler()
    sequential_scaler = MinMaxScaler()

    age_values = data.x_static_raw[indices, age_index].reshape(-1, 1)
    age_scaler.fit(age_values)

    seq_subset = data.x_sequential_raw[indices]
    mask_subset = data.mask[indices]
    valid_seq_values = seq_subset[mask_subset].reshape(-1, 1)
    if len(valid_seq_values) == 0:
        raise ValueError("No valid sequential values available for scaler fitting.")
    sequential_scaler.fit(valid_seq_values)

    return FittedScalers(
        age_scaler=age_scaler,
        sequential_scaler=sequential_scaler,
        age_index=age_index,
    )


def transform_subset(
    data: PatientArrays,
    indices: np.ndarray,
    scalers: FittedScalers,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    x_static = data.x_static_raw[indices].copy()
    x_seq = np.zeros_like(data.x_sequential_raw[indices], dtype=np.float32)
    y_seq = data.y_sequence[indices].copy().astype(np.float32)
    mask = data.mask[indices].copy()
    y_overall = data.y_overall[indices].copy().astype(np.float32)

    x_static[:, scalers.age_index] = scalers.age_scaler.transform(
        x_static[:, scalers.age_index].reshape(-1, 1)
    ).ravel().astype(np.float32)

    raw_seq = data.x_sequential_raw[indices]
    if mask.any():
        scaled_values = scalers.sequential_scaler.transform(
            raw_seq[mask].reshape(-1, 1)
        ).ravel().astype(np.float32)
        x_seq[mask] = scaled_values.reshape(-1, 1)

    return x_static, x_seq, y_seq, mask, y_overall


# -----------------------------------------------------------------------------
# PyTorch dataset/model
# -----------------------------------------------------------------------------

class PatientDataset(Dataset):
    def __init__(
        self,
        x_static: np.ndarray,
        x_sequential: np.ndarray,
        y_sequence: np.ndarray,
        mask: np.ndarray,
        y_overall: np.ndarray,
    ) -> None:
        self.x_static = torch.as_tensor(x_static, dtype=torch.float32)
        self.x_sequential = torch.as_tensor(x_sequential, dtype=torch.float32)
        self.y_sequence = torch.as_tensor(y_sequence, dtype=torch.float32)
        self.mask = torch.as_tensor(mask, dtype=torch.bool)
        self.y_overall = torch.as_tensor(y_overall, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.y_overall)

    def __getitem__(self, idx: int):
        return (
            self.x_static[idx],
            self.x_sequential[idx],
            self.y_sequence[idx],
            self.mask[idx],
            self.y_overall[idx],
        )


class LSTMAttention(nn.Module):
    """Attention-based LSTM with static-temporal fusion."""

    def __init__(self, static_dim: int, sequential_dim: int, hidden_dim: int, num_layers: int):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=sequential_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.attention_layer = nn.Linear(hidden_dim, 1)
        self.combined_layer = nn.Linear(hidden_dim + static_dim, hidden_dim)
        self.fc_overall = nn.Linear(hidden_dim, 1)
        self.fc_timestep = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def attention_net(
        self, lstm_output: torch.Tensor, mask: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        attention_scores = self.attention_layer(lstm_output).squeeze(-1)  # (B,T)
        attention_scores = attention_scores.masked_fill(~mask, -1e9)
        attention_weights = torch.softmax(attention_scores, dim=1).unsqueeze(-1)  # (B,T,1)
        weighted_output = attention_weights * lstm_output
        return weighted_output, attention_weights

    def forward(
        self, x_static: torch.Tensor, x_sequential: torch.Tensor, mask: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        lstm_output, _ = self.lstm(x_sequential)
        weighted_output, attention_weights = self.attention_net(lstm_output, mask)

        # Per-time-step prediction head
        timestep_prob = self.sigmoid(self.fc_timestep(weighted_output)).squeeze(-1)  # (B,T)

        context_vector = torch.sum(weighted_output, dim=1)  # (B,H)
        combined_features = torch.cat((context_vector, x_static), dim=1)
        combined_out = torch.relu(self.combined_layer(combined_features))
        overall_prob = self.sigmoid(self.fc_overall(combined_out)).squeeze(-1)  # (B,)
        return overall_prob, timestep_prob, attention_weights.squeeze(-1)


def make_loader(
    x_static: np.ndarray,
    x_seq: np.ndarray,
    y_seq: np.ndarray,
    mask: np.ndarray,
    y_overall: np.ndarray,
    batch_size: int,
    shuffle: bool,
    seed: int = SEED,
) -> DataLoader:
    dataset = PatientDataset(x_static, x_seq, y_seq, mask, y_overall)
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        generator=generator if shuffle else None,
    )


def train_model(
    x_static: np.ndarray,
    x_seq: np.ndarray,
    y_seq: np.ndarray,
    mask: np.ndarray,
    y_overall: np.ndarray,
    config: ModelConfig,
    device: torch.device,
    verbose: bool = True,
    seed: int = SEED,
) -> Tuple[LSTMAttention, pd.DataFrame]:
    set_seed(seed)
    loader = make_loader(
        x_static, x_seq, y_seq, mask, y_overall,
        batch_size=config.batch_size,
        shuffle=True,
        seed=seed,
    )

    model = LSTMAttention(
        static_dim=x_static.shape[1],
        sequential_dim=x_seq.shape[2],
        hidden_dim=config.hidden_dim,
        num_layers=config.num_layers,
    ).to(device)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    history = []
    for epoch in range(config.epochs):
        model.train()
        running_loss = 0.0
        n_batches = 0

        for x_static_b, x_seq_b, y_seq_b, mask_b, y_overall_b in loader:
            x_static_b = x_static_b.to(device)
            x_seq_b = x_seq_b.to(device)
            y_seq_b = y_seq_b.to(device)
            mask_b = mask_b.to(device)
            y_overall_b = y_overall_b.to(device)

            optimizer.zero_grad(set_to_none=True)
            overall_prob, timestep_prob, _ = model(x_static_b, x_seq_b, mask_b)

            # Correct patient-level target: any AIP in the valid modeled horizon.
            overall_loss = criterion(overall_prob, y_overall_b)

            # Masked per-window loss: no padded or post-event windows can contribute.
            valid_timestep_prob = timestep_prob[mask_b]
            valid_timestep_labels = y_seq_b[mask_b]
            timestep_loss = criterion(valid_timestep_prob, valid_timestep_labels)

            loss = overall_loss + timestep_loss
            loss.backward()
            optimizer.step()

            running_loss += float(loss.item())
            n_batches += 1

        epoch_loss = running_loss / max(n_batches, 1)
        history.append({"epoch": epoch + 1, "loss": epoch_loss})
        if verbose:
            print(f"Epoch {epoch + 1:02d}/{config.epochs}: loss={epoch_loss:.6f}")

    return model, pd.DataFrame(history)


# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------

def overall_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_true_i = y_true.astype(int)
    y_pred = (y_prob >= threshold).astype(int)

    cm = confusion_matrix(y_true_i, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true_i, y_pred, average="weighted", zero_division=0
    )

    metrics = {
        "n": float(len(y_true_i)),
        "n_positive": float(y_true_i.sum()),
        "prevalence": float(y_true_i.mean()),
        "threshold": float(threshold),
        "auroc": float(roc_auc_score(y_true_i, y_prob)),
        "auprc": float(average_precision_score(y_true_i, y_prob)),
        "accuracy": float(accuracy_score(y_true_i, y_pred)),
        "sensitivity": float(recall_score(y_true_i, y_pred, zero_division=0)),
        "specificity": float(specificity),
        "precision": float(precision_score(y_true_i, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true_i, y_pred, zero_division=0)),
        "weighted_precision": float(weighted_precision),
        "weighted_recall": float(weighted_recall),
        "weighted_f1": float(weighted_f1),
        "tn": float(tn),
        "fp": float(fp),
        "fn": float(fn),
        "tp": float(tp),
    }
    return metrics


def make_curve_tables(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    calibration_bins: int = 10,
    calibration_strategy: str = "uniform",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    fpr, tpr, roc_thresholds = roc_curve(y_true.astype(int), y_prob)
    roc_table = pd.DataFrame(
        {"fpr": fpr, "tpr": tpr, "threshold": roc_thresholds}
    )

    precision, recall, pr_thresholds = precision_recall_curve(y_true.astype(int), y_prob)
    # sklearn returns one fewer threshold than precision/recall points.
    pr_threshold_col = np.r_[pr_thresholds, np.nan]
    pr_table = pd.DataFrame(
        {"recall": recall, "precision": precision, "threshold": pr_threshold_col}
    )

    prob_true, prob_pred = calibration_curve(
        y_true.astype(int),
        y_prob,
        n_bins=calibration_bins,
        strategy=calibration_strategy,
    )
    calibration_table = pd.DataFrame(
        {"mean_predicted_probability": prob_pred, "observed_event_rate": prob_true}
    )
    return roc_table, pr_table, calibration_table


def compute_window_metrics(
    y_sequence: np.ndarray,
    timestep_prob: np.ndarray,
    mask: np.ndarray,
    raw_exposure: np.ndarray,
    threshold: float,
) -> pd.DataFrame:
    rows = []
    for t in range(y_sequence.shape[1]):
        valid = mask[:, t]
        n_valid = int(valid.sum())
        if n_valid == 0:
            rows.append(
                {
                    "window": t + 1,
                    "n_at_risk_windows": 0,
                    "n_aip": 0,
                    "aip_incidence": np.nan,
                    "mean_exposure_aip": np.nan,
                    "mean_exposure_non_aip": np.nan,
                    "accuracy": np.nan,
                    "precision": np.nan,
                    "recall": np.nan,
                    "f1": np.nan,
                    "tn": np.nan,
                    "fp": np.nan,
                    "fn": np.nan,
                    "tp": np.nan,
                }
            )
            continue

        y_true_t = y_sequence[valid, t].astype(int)
        y_prob_t = timestep_prob[valid, t]
        y_pred_t = (y_prob_t >= threshold).astype(int)
        exposure_t = raw_exposure[valid, t, 0]

        cm = confusion_matrix(y_true_t, y_pred_t, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        event_exposure = exposure_t[y_true_t == 1]
        nonevent_exposure = exposure_t[y_true_t == 0]

        rows.append(
            {
                "window": t + 1,
                "n_at_risk_windows": n_valid,
                "n_aip": int(y_true_t.sum()),
                "aip_incidence": float(y_true_t.mean()),
                "mean_exposure_aip": float(event_exposure.mean()) if len(event_exposure) else np.nan,
                "mean_exposure_non_aip": float(nonevent_exposure.mean()) if len(nonevent_exposure) else np.nan,
                "accuracy": float(accuracy_score(y_true_t, y_pred_t)),
                "precision": float(precision_score(y_true_t, y_pred_t, zero_division=0)),
                "recall": float(recall_score(y_true_t, y_pred_t, zero_division=0)),
                "f1": float(f1_score(y_true_t, y_pred_t, zero_division=0)),
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
            }
        )
    return pd.DataFrame(rows)


def evaluate_model(
    model: LSTMAttention,
    x_static: np.ndarray,
    x_seq: np.ndarray,
    y_seq: np.ndarray,
    mask: np.ndarray,
    y_overall: np.ndarray,
    raw_exposure: np.ndarray,
    threshold: float,
    config: ModelConfig,
    device: torch.device,
    calibration_bins: int = 10,
    calibration_strategy: str = "uniform",
) -> EvaluationResult:
    loader = make_loader(
        x_static, x_seq, y_seq, mask, y_overall,
        batch_size=config.batch_size,
        shuffle=False,
    )

    model.eval()
    all_overall_prob = []
    all_timestep_prob = []
    all_y_overall = []
    all_y_seq = []
    all_mask = []

    with torch.no_grad():
        for x_static_b, x_seq_b, y_seq_b, mask_b, y_overall_b in loader:
            x_static_b = x_static_b.to(device)
            x_seq_b = x_seq_b.to(device)
            mask_device = mask_b.to(device)

            overall_prob, timestep_prob, _ = model(x_static_b, x_seq_b, mask_device)
            all_overall_prob.append(overall_prob.cpu().numpy())
            all_timestep_prob.append(timestep_prob.cpu().numpy())
            all_y_overall.append(y_overall_b.numpy())
            all_y_seq.append(y_seq_b.numpy())
            all_mask.append(mask_b.numpy())

    y_prob = np.concatenate(all_overall_prob)
    timestep_prob = np.concatenate(all_timestep_prob)
    y_true = np.concatenate(all_y_overall)
    y_sequence_all = np.concatenate(all_y_seq)
    mask_all = np.concatenate(all_mask).astype(bool)
    y_pred = (y_prob >= threshold).astype(int)

    metrics = overall_metrics(y_true, y_prob, threshold=threshold)
    window_metrics = compute_window_metrics(
        y_sequence_all,
        timestep_prob,
        mask_all,
        raw_exposure=raw_exposure,
        threshold=threshold,
    )
    roc_table, pr_table, calibration_table = make_curve_tables(
        y_true,
        y_prob,
        calibration_bins=calibration_bins,
        calibration_strategy=calibration_strategy,
    )

    return EvaluationResult(
        metrics=metrics,
        y_true=y_true,
        y_prob=y_prob,
        y_pred=y_pred,
        timestep_prob=timestep_prob,
        y_sequence=y_sequence_all,
        mask=mask_all,
        window_metrics=window_metrics,
        roc_table=roc_table,
        pr_table=pr_table,
        calibration_table=calibration_table,
    )


# -----------------------------------------------------------------------------
# Five-fold CV / optional hyperparameter search
# -----------------------------------------------------------------------------

def load_cv_grid(path: Optional[str], default_config: ModelConfig) -> List[ModelConfig]:
    if path is None:
        return [default_config]

    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    if not isinstance(payload, list) or not payload:
        raise ValueError("--cv-grid-json must contain a non-empty JSON list of configuration objects.")

    configs = []
    allowed = set(asdict(default_config).keys())
    for i, item in enumerate(payload):
        if not isinstance(item, dict):
            raise ValueError(f"CV candidate {i} is not a JSON object.")
        unknown = set(item) - allowed
        if unknown:
            raise ValueError(f"Unknown CV config keys in candidate {i}: {sorted(unknown)}")
        configs.append(replace(default_config, **item))
    return configs


def run_cross_validation(
    data: PatientArrays,
    train_indices: np.ndarray,
    candidate_configs: Sequence[ModelConfig],
    device: torch.device,
    n_splits: int = 5,
    seed: int = SEED,
) -> Tuple[ModelConfig, pd.DataFrame]:
    train_y = data.y_overall[train_indices].astype(int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    all_rows = []
    for candidate_id, config in enumerate(candidate_configs, start=1):
        print("\n" + "=" * 80)
        print(f"CV candidate {candidate_id}/{len(candidate_configs)}: {asdict(config)}")
        print("=" * 80)

        for fold, (fold_train_pos, fold_val_pos) in enumerate(skf.split(train_indices, train_y), start=1):
            fold_train_idx = train_indices[fold_train_pos]
            fold_val_idx = train_indices[fold_val_pos]

            scalers = fit_scalers(data, fold_train_idx)
            tr = transform_subset(data, fold_train_idx, scalers)
            va = transform_subset(data, fold_val_idx, scalers)

            print(f"\nCandidate {candidate_id}, fold {fold}/{n_splits}")
            model, _ = train_model(
                *tr,
                config=config,
                device=device,
                verbose=False,
                seed=seed + candidate_id * 100 + fold,
            )

            # CV ranking is threshold-free: AUROC primary, AUPRC secondary.
            fold_threshold = float(data.y_overall[fold_train_idx].mean())
            result = evaluate_model(
                model,
                *va,
                raw_exposure=data.x_sequential_raw[fold_val_idx],
                threshold=fold_threshold,
                config=config,
                device=device,
            )

            row = {
                "candidate_id": candidate_id,
                "fold": fold,
                **asdict(config),
                "fold_train_n": len(fold_train_idx),
                "fold_val_n": len(fold_val_idx),
                "fold_threshold_train_incidence": fold_threshold,
                "val_auroc": result.metrics["auroc"],
                "val_auprc": result.metrics["auprc"],
                "val_accuracy_at_incidence_threshold": result.metrics["accuracy"],
            }
            all_rows.append(row)
            print(
                f"val AUROC={row['val_auroc']:.4f}, "
                f"AUPRC={row['val_auprc']:.4f}, "
                f"accuracy={row['val_accuracy_at_incidence_threshold']:.4f}"
            )

            # Release fold model promptly on GPU systems.
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    cv_df = pd.DataFrame(all_rows)
    summary = (
        cv_df.groupby("candidate_id", as_index=False)
        .agg(mean_auroc=("val_auroc", "mean"), mean_auprc=("val_auprc", "mean"))
        .sort_values(["mean_auroc", "mean_auprc"], ascending=False)
    )
    best_candidate_id = int(summary.iloc[0]["candidate_id"])
    best_config = candidate_configs[best_candidate_id - 1]

    print("\nCV summary:")
    print(summary.to_string(index=False))
    print(f"Selected configuration: candidate {best_candidate_id} -> {asdict(best_config)}")

    return best_config, cv_df


# -----------------------------------------------------------------------------
# Outputs / plots
# -----------------------------------------------------------------------------

def save_json(data: dict, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def plot_roc(result: EvaluationResult, output_path: Path, label: str) -> None:
    plt.figure(figsize=(6, 5))
    plt.plot(result.roc_table["fpr"], result.roc_table["tpr"], label=f"{label} AUROC={result.metrics['auroc']:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


def plot_pr(result: EvaluationResult, output_path: Path, label: str) -> None:
    plt.figure(figsize=(6, 5))
    plt.plot(result.pr_table["recall"], result.pr_table["precision"], label=f"{label} AUPRC={result.metrics['auprc']:.4f}")
    plt.axhline(result.metrics["prevalence"], linestyle="--", label=f"Prevalence={result.metrics['prevalence']:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


def plot_calibration(result: EvaluationResult, output_path: Path, label: str) -> None:
    plt.figure(figsize=(6, 5))
    plt.plot(
        result.calibration_table["mean_predicted_probability"],
        result.calibration_table["observed_event_rate"],
        marker="o",
        label=label,
    )
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed AIP event rate")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


def plot_window_accuracy(result: EvaluationResult, output_path: Path, overall_accuracy: float) -> None:
    df = result.window_metrics
    mean_accuracy = float(df["accuracy"].mean(skipna=True))
    plt.figure(figsize=(8, 5))
    plt.plot(df["window"], df["accuracy"], marker="o")
    plt.axhline(overall_accuracy, linestyle="--", label=f"Overall accuracy={overall_accuracy:.4f}")
    plt.axhline(mean_accuracy, linestyle=":", label=f"Mean window accuracy={mean_accuracy:.4f}")
    plt.xticks(range(1, MAX_WINDOWS + 1))
    plt.xlabel("30-day follow-up window")
    plt.ylabel("Accuracy")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


def save_evaluation(result: EvaluationResult, output_dir: Path, prefix: str, label: str) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    save_json(result.metrics, output_dir / f"{prefix}_metrics.json")
    result.window_metrics.to_csv(output_dir / f"{prefix}_window_metrics.csv", index=False)
    result.roc_table.to_csv(output_dir / f"{prefix}_roc_curve.csv", index=False)
    result.pr_table.to_csv(output_dir / f"{prefix}_pr_curve.csv", index=False)
    result.calibration_table.to_csv(output_dir / f"{prefix}_calibration.csv", index=False)
    pd.DataFrame(
        {
            "y_true": result.y_true.astype(int),
            "predicted_probability": result.y_prob,
            "predicted_class": result.y_pred.astype(int),
        }
    ).to_csv(output_dir / f"{prefix}_patient_predictions.csv", index=False)

    plot_roc(result, output_dir / f"{prefix}_roc.png", label=label)
    plot_pr(result, output_dir / f"{prefix}_pr.png", label=label)
    plot_calibration(result, output_dir / f"{prefix}_calibration.png", label=label)
    plot_window_accuracy(
        result,
        output_dir / f"{prefix}_window_accuracy.png",
        overall_accuracy=result.metrics["accuracy"],
    )


def print_metrics(title: str, metrics: Dict[str, float]) -> None:
    print("\n" + title)
    print("-" * len(title))
    keys = [
        "n", "n_positive", "prevalence", "threshold", "auroc", "auprc",
        "accuracy", "sensitivity", "specificity", "precision", "f1",
        "weighted_precision", "weighted_recall", "weighted_f1",
        "tn", "fp", "fn", "tp",
    ]
    for key in keys:
        value = metrics[key]
        if key in {"n", "n_positive", "tn", "fp", "fn", "tp"}:
            print(f"{key:>20s}: {int(value):,}")
        else:
            print(f"{key:>20s}: {value:.6f}")




## 2. Analysis
Set the input and output paths, then run the cell.


In [ ]:
# ==============================
# Analysis configuration (edit here)
# ==============================
DEVELOPMENT_SAS = "/content/dip_cohort.sas7bdat"  # change to your file path
OUTPUT_DIR = "/content/dip_model_output"

# If sequence-length harmonization was already performed in SAS, keep False.
HARMONIZE_NON_EVENTS = False

# Five-fold cross-validation
SKIP_CV = False

# Optional hyperparameter grid
CV_GRID_JSON = None

# Internal threshold: incidence in the 80% training cohort.
# This avoids selecting the cutoff from held-out test labels.
INTERNAL_THRESHOLD_MODE = "train_incidence"  # train_incidence / development_incidence / fixed
INTERNAL_FIXED_THRESHOLD = 0.0168

CALIBRATION_BINS = 10
CALIBRATION_STRATEGY = "uniform"  # uniform / quantile


# ==============================
# Run internal development + held-out test analysis
# ==============================
set_seed(SEED)
device = get_device()
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"PyTorch device: {device}")
print(f"Random seed: {SEED}")

dev_df = load_table(DEVELOPMENT_SAS)
dev_df, static_features = preprocess_dataframe(dev_df)
dev_data = build_patient_arrays(
    dev_df,
    static_features=static_features,
    harmonize_non_events=HARMONIZE_NON_EVENTS,
    seed=SEED,
)

seq_summary = sequence_length_summary(dev_data)
seq_summary.to_csv(output_dir / "development_sequence_length_distribution.csv", index=False)

print(f"\nDevelopment patients: {len(dev_data):,}")
print(f"Development AIP events: {int(dev_data.y_overall.sum()):,}")
print(f"Development incidence: {dev_data.y_overall.mean():.6f}")
print(f"Static features ({len(static_features)}): {static_features}")

patient_indices = np.arange(len(dev_data))
train_idx, test_idx = train_test_split(
    patient_indices,
    test_size=0.20,
    random_state=SEED,
    stratify=dev_data.y_overall.astype(int),
)

print(f"Train patients: {len(train_idx):,}; AIP incidence={dev_data.y_overall[train_idx].mean():.6f}")
print(f"Held-out test patients: {len(test_idx):,}; AIP incidence={dev_data.y_overall[test_idx].mean():.6f}")

default_config = ModelConfig()
candidate_configs = load_cv_grid(CV_GRID_JSON, default_config)

if SKIP_CV:
    warnings.warn("Five-fold CV was skipped.")
    selected_config = default_config
    cv_df = pd.DataFrame()
else:
    selected_config, cv_df = run_cross_validation(
        dev_data,
        train_indices=train_idx,
        candidate_configs=candidate_configs,
        device=device,
        n_splits=5,
        seed=SEED,
    )
    cv_df.to_csv(output_dir / "five_fold_cv_results.csv", index=False)

# Fit preprocessing only on the 80% training set.
final_scalers = fit_scalers(dev_data, train_idx)
train_data = transform_subset(dev_data, train_idx, final_scalers)
test_data = transform_subset(dev_data, test_idx, final_scalers)

print("\nTraining final model on the full 80% training set...")
final_model, history = train_model(
    *train_data,
    config=selected_config,
    device=device,
    verbose=True,
    seed=SEED,
)
history.to_csv(output_dir / "final_training_history.csv", index=False)

train_incidence = float(dev_data.y_overall[train_idx].mean())
development_incidence = float(dev_data.y_overall.mean())
if INTERNAL_THRESHOLD_MODE == "train_incidence":
    internal_threshold = train_incidence
elif INTERNAL_THRESHOLD_MODE == "development_incidence":
    internal_threshold = development_incidence
elif INTERNAL_THRESHOLD_MODE == "fixed":
    internal_threshold = float(INTERNAL_FIXED_THRESHOLD)
else:
    raise ValueError("INTERNAL_THRESHOLD_MODE must be train_incidence, development_incidence, or fixed")

test_result = evaluate_model(
    final_model,
    *test_data,
    raw_exposure=dev_data.x_sequential_raw[test_idx],
    threshold=internal_threshold,
    config=selected_config,
    device=device,
    calibration_bins=CALIBRATION_BINS,
    calibration_strategy=CALIBRATION_STRATEGY,
)
print_metrics("Held-out internal test performance", test_result.metrics)
save_evaluation(test_result, output_dir, prefix="internal_test", label="Internal test")

# Save model + preprocessing objects.
torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "model_config": asdict(selected_config),
        "static_features": static_features,
        "max_windows": MAX_WINDOWS,
        "sequential_feature": SEQUENTIAL_COL,
        "internal_threshold": internal_threshold,
        "internal_threshold_mode": INTERNAL_THRESHOLD_MODE,
    },
    output_dir / "final_model.pt",
)
with open(output_dir / "preprocessing_scalers.pkl", "wb") as f:
    pickle.dump(final_scalers, f)

run_metadata = {
    "seed": SEED,
    "device": str(device),
    "max_windows": MAX_WINDOWS,
    "static_features": static_features,
    "harmonize_non_events_in_python": bool(HARMONIZE_NON_EVENTS),
    "selected_model_config": asdict(selected_config),
    "development_n": int(len(dev_data)),
    "development_events": int(dev_data.y_overall.sum()),
    "development_incidence": development_incidence,
    "train_n": int(len(train_idx)),
    "train_incidence": train_incidence,
    "test_n": int(len(test_idx)),
    "test_incidence": float(dev_data.y_overall[test_idx].mean()),
    "internal_threshold": internal_threshold,
    "internal_threshold_mode": INTERNAL_THRESHOLD_MODE,
    "calibration_bins": int(CALIBRATION_BINS),
    "calibration_strategy": CALIBRATION_STRATEGY,
}
save_json(run_metadata, output_dir / "run_metadata.json")

print(f"\nDone. Outputs saved to: {output_dir.resolve()}")


## 3. Save outputs


In [ ]:
import shutil
zip_path = shutil.make_archive('/content/dip_model_output', 'zip', OUTPUT_DIR)
print(zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass
